# Target Data Reconciliation: Sales vs Orders
**Datathon 2026 - Data Integrity Audit**

This notebook verifies the consistency between the raw `order_items` data and the pre-aggregated `sales.csv`. This ensures we are training on the correct target variable.

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Project configuration
PROCESSED_DIR = Path("../data/processed")
PLOT_DIR = Path("../data/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.options.display.max_columns = None

## 1. Loading Aggregated vs Raw
Comparing `sales.csv` summary with `order_items` aggregations.

In [4]:
print("Loading datasets...")
sales_agg = pd.read_parquet(PROCESSED_DIR / "sales.parquet")
order_items = pd.read_parquet(PROCESSED_DIR / "order_items.parquet")
orders = pd.read_parquet(PROCESSED_DIR / "orders.parquet")

# Process Raw Data (Order Items)
order_items = order_items.merge(orders[['order_id', 'order_date']], on='order_id')
order_items['revenue_calc'] = order_items['quantity'] * order_items['unit_price']
order_items['order_date'] = pd.to_datetime(order_items['order_date'])
order_items['year'] = order_items['order_date'].dt.year
order_items['month'] = order_items['order_date'].dt.month

raw_monthly = order_items.groupby(['year', 'month'])['revenue_calc'].sum().reset_index(name='raw_revenue')

# Process Aggregated Data (Sales.csv)
sales_agg['Date'] = pd.to_datetime(sales_agg['Date'])
sales_agg['year'] = sales_agg['Date'].dt.year
sales_agg['month'] = sales_agg['Date'].dt.month

agg_monthly = sales_agg.groupby(['year', 'month'])['Revenue'].sum().reset_index(name='agg_revenue')

# Merge for comparison
comparison = raw_monthly.merge(agg_monthly, on=['year', 'month'])
comparison['year_month'] = comparison['year'].astype(str) + '-' + comparison['month'].astype(str).str.zfill(2)

print("Reconciliation summary:")
print(comparison.head())
print(f"Revenue gap: {(comparison['raw_revenue'] - comparison['agg_revenue']).sum():.2f}")

Loading datasets...
Reconciliation summary:
   year  month   raw_revenue   agg_revenue year_month
0  2012      7  1.304068e+08  1.304068e+08    2012-07
1  2012      8  1.590892e+08  1.590892e+08    2012-08
2  2012      9  1.293071e+08  1.293071e+08    2012-09
3  2012     10  1.101857e+08  1.101857e+08    2012-10
4  2012     11  9.818630e+07  9.818630e+07    2012-11
Revenue gap: -0.00


## 2. Visual Reconciliation
Do the trends match perfectly?

In [5]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=comparison['year_month'], y=comparison['raw_revenue'],
                         mode='lines', name='Revenue (Order Items)', line=dict(color='blue', width=4)))

fig.add_trace(go.Scatter(x=comparison['year_month'], y=comparison['agg_revenue'],
                         mode='lines', name='Revenue (Sales Summary)', line=dict(color='yellow', dash='dash')))

fig.update_layout(title='Audit: Sales Summary vs Order Items Reconciliation')
fig.write_image(PLOT_DIR / "21_target_reconciliation.png")
fig.show()

## 3. Findings Summary
- If the lines overlap perfectly, the `order_items` dataset is valid for training models.
- If there's a gap, check for hidden fees, returns, or missing orders in the `order_items` table.